# HYG−LQD 信用风险偏好：Stage1 beta上限释放局部价值信息门

前一交易日完整20日来源、相同239/301物理0bp Tiered受限标签；旧H5行情预测路线关闭，仅检验不同接收任务。


In [ ]:
import hashlib,json,os,pathlib,shlex
import numpy as np,pandas as pd,qlib
from qlib.data.dataset import DatasetH
from qlib.data.dataset.handler import DataHandlerLP
from qlib.workflow import R
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from src.analysis.seed_disagreement_information import full_calendar_bootstrap
from src.data.nas_market_data import archive_nas_recorder,_settings,_ssh
root=pathlib.Path.cwd();card=root/'research/stage1_credit_pair_cap_local_value_information_20261023';plan=json.loads((card/'plan.json').read_text());src=plan['sources'];sha=lambda p:hashlib.sha256(pathlib.Path(p).read_bytes()).hexdigest();assert plan['frozen'] and plan['execution_authorized_by_active_Goal']
credit_card=root/src['credit_card'];control_card=root/src['control_card'];label_card=root/src['label_card']
assert sha(credit_card/'summary.json')==src['credit_summary_sha256'] and sha(control_card/'summary.json')==src['control_summary_sha256'] and sha(label_card/'summary.json')==src['label_summary_sha256'] and sha(root/'src/analysis/seed_disagreement_information.py')==src['bootstrap_module_sha256']
assert sha(credit_card/'qlib_recorder'/src['credit_experiment_id']/src['credit_recorder_id']/'artifacts/analysis/mapped_signals')==src['credit_map_artifact_sha256']
assert sha(control_card/'qlib_recorder'/src['control_experiment_id']/src['control_recorder_id']/'artifacts/analysis/mapped_source')==src['control_map_artifact_sha256']
assert sha(label_card/'qlib_recorder'/src['label_experiment_id']/src['label_recorder_id']/'artifacts/analysis/labels')==src['label_artifact_sha256']
os.environ['MLFLOW_ALLOW_FILE_STORE']='true';qlib.init(provider_uri=str(root/'data/qlib/minute_information_pit_allocator_20261012/daily_prices'),region='us',kernels=1)
with R.uri_context(str(credit_card/'qlib_recorder')):
 rec=R.get_recorder(experiment_id=src['credit_experiment_id'],recorder_id=src['credit_recorder_id']);assert rec.status=='FINISHED';credit=rec.load_object('analysis/mapped_signals')
with R.uri_context(str(control_card/'qlib_recorder')):
 rec=R.get_recorder(experiment_id=src['control_experiment_id'],recorder_id=src['control_recorder_id']);assert rec.status=='FINISHED';mapped=rec.load_object('analysis/mapped_source').reset_index(drop=True)
with R.uri_context(str(label_card/'qlib_recorder')):
 rec=R.get_recorder(experiment_id=src['label_experiment_id'],recorder_id=src['label_recorder_id']);assert rec.status=='FINISHED';labels=rec.load_object('analysis/labels').reset_index(drop=True)
assert len(mapped)==len(labels)==1061 and mapped.trade_date.equals(labels.trade_date) and mapped.signal_date.equals(labels.signal_date)
assert mapped.eligible.equals(labels.eligible) and mapped.loc[mapped.eligible,'raw_beta_minus_cap'].gt(1e-8).all()
assert (pd.to_datetime(mapped.signal_date)<pd.to_datetime(mapped.trade_date)).all()
credit=credit.copy();credit.index=pd.to_datetime(credit.index)
source_rows=credit.reindex(pd.to_datetime(mapped.signal_date));assert source_rows.index.is_unique and source_rows.source_known.all() and len(source_rows)==1061
mapped=mapped.drop(columns=['log_VIX9D_over_VIX3M_prev','known','source_session']).copy()
mapped['HYG_minus_LQD_log_total_return_20']=source_rows.HYG_minus_LQD_log_total_return_20.to_numpy(float)
mapped['known']=source_rows.source_known.to_numpy(bool)
assert np.isfinite(mapped.HYG_minus_LQD_log_total_return_20.to_numpy(float)).all()
features=plan['receiver']['baseline']+[plan['receiver']['candidate_extra']];flat=mapped[['trade_date','signal_date',*features,'known','eligible','cap_binding','label_available_at']].copy();flat['local_fee_net_advantage']=labels.local_fee_net_advantage.astype(float)
static=pd.concat({'feature':flat[features],'label':flat[['local_fee_net_advantage']],'meta':flat[['known','eligible','cap_binding','label_available_at']]},axis=1);static.index=pd.MultiIndex.from_arrays([flat.trade_date,['SOXL_SOXX_CASH']*len(flat)],names=['datetime','instrument']);dataset=DatasetH(handler=DataHandlerLP.from_df(static),segments={'train':('2018-10-15','2020-12-31'),'evaluation':('2021-01-01','2022-12-30')})
tr=dataset.prepare('train',col_set=['feature','label','meta']);ev=dataset.prepare('evaluation',col_set=['feature','label','meta']);train_cutoff=pd.Timestamp('2021-01-01T00:00:00',tz='America/New_York').tz_convert('UTC');eval_cutoff=pd.Timestamp('2023-01-01T00:00:00',tz='America/New_York').tz_convert('UTC');tr_dates=pd.DatetimeIndex(tr.index.get_level_values('datetime'));ev_dates=pd.DatetimeIndex(ev.index.get_level_values('datetime'))
train_mask=tr[('meta','known')].to_numpy(bool)&tr[('meta','eligible')].to_numpy(bool)&tr[('meta','cap_binding')].to_numpy(bool)&pd.to_datetime(tr[('meta','label_available_at')],utc=True).lt(train_cutoff).to_numpy()&np.isfinite(tr[('label','local_fee_net_advantage')].to_numpy(float))
eval_mask=ev[('meta','known')].to_numpy(bool)&ev[('meta','eligible')].to_numpy(bool)&ev[('meta','cap_binding')].to_numpy(bool)&pd.to_datetime(ev[('meta','label_available_at')],utc=True).lt(eval_cutoff).to_numpy()&np.isfinite(ev[('label','local_fee_net_advantage')].to_numpy(float))
train=tr.iloc[np.flatnonzero(train_mask)];evaluation=ev.iloc[np.flatnonzero(eval_mask)];dates=pd.DatetimeIndex(evaluation.index.get_level_values('datetime'));coh=plan['cohorts'];assert len(train)==coh['train_mature_binding']==239 and len(evaluation)==coh['evaluation_mature_binding']==301 and len(ev)==coh['evaluation_full_calendar']==503 and int((dates.year==2021).sum())==56 and int((dates.year==2022).sum())==245
base=plan['receiver']['baseline'];xtr=train['feature'][base].to_numpy(float);xev=evaluation['feature'][base].to_numpy(float);xtr_new=train['feature'][features].to_numpy(float);xev_new=evaluation['feature'][features].to_numpy(float);ytr=train[('label','local_fee_net_advantage')].to_numpy(float);yev=evaluation[('label','local_fee_net_advantage')].to_numpy(float);assert np.isfinite(xtr).all() and np.isfinite(xtr_new).all() and np.isfinite(yev).all()
preflight={'status':'fixed_credit_pair_source_and_cap_local_value_cohort_matched','plan_sha256':sha(card/'plan.json'),'full_OOF_calendar':len(mapped),'train_mature_binding':len(train),'evaluation_mature_binding':len(evaluation),'evaluation_2021':56,'evaluation_2022':245,'source_Recorder_object_loads':3,'model_fits':0,'native_accounts':0};(card/'preflight_result.json').write_text(json.dumps(preflight,indent=2)+'\n')
state={'status':'fit_started','plan_sha256':preflight['plan_sha256'],'fit_attempts':0,'fit_completed':0,'bootstrap_attempts':0,'native_accounts':0};(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
control=make_pipeline(SimpleImputer(strategy='median'),StandardScaler(),Ridge(alpha=1.0));candidate=make_pipeline(SimpleImputer(strategy='median'),StandardScaler(),Ridge(alpha=1.0))
for model,x in ((control,xtr),(candidate,xtr_new)):
 state['fit_attempts']+=1;(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n');model.fit(x,ytr);state['fit_completed']+=1;(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
p0=control.predict(xev);p1=candidate.predict(xev_new);mean=float(ytr.mean());coef=float(candidate.named_steps['ridge'].coef_[-1]);assert abs(float(np.square(yev-p0).mean())-6.368773792168581e-06)<1e-12 and abs(float(np.square(yev-mean).mean())-6.330007046203221e-06)<1e-12
scored=pd.DataFrame({'trade_date':dates,'label':yev,'baseline_prediction':p0,'candidate_prediction':p1,'train_mean_prediction':mean});scored['gain_vs_baseline']=(yev-p0)**2-(yev-p1)**2;scored['gain_vs_train_mean']=(yev-mean)**2-(yev-p1)**2
full=pd.DataFrame({'trade_date':ev_dates}).merge(scored,on='trade_date',how='left',validate='one_to_one');assert len(full)==503 and int(full.label.notna().sum())==301
metrics={'candidate_MSE':float(np.square(yev-p1).mean()),'baseline_MSE':float(np.square(yev-p0).mean()),'train_mean_MSE':float(np.square(yev-mean).mean()),'added_standardized_coefficient':coef,'comparators':{}};gates={'coefficient_positive':bool(coef>0)};reps={};indices=None
for key in ('baseline','train_mean'):
 column='gain_vs_'+key;diff=scored[column];annual=[]
 for year in (2021,2022):
  take=scored.trade_date.dt.year.eq(year);actual=scored.loc[take,'label'];pred=scored.loc[take,'candidate_prediction'];rho=float(pd.Series(pred.to_numpy(float)).corr(pd.Series(actual.to_numpy(float)),method='spearman'));annual.append({'year':year,'rows':int(take.sum()),'mean_gain':float(diff.loc[take].mean()),'Spearman':rho})
 state['bootstrap_attempts']+=1;(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n');bp=plan['information_gate']['bootstrap'];interval,idx,rep=full_calendar_bootstrap(full[column].to_numpy(float),mean_block=bp['mean_block_sessions'],resamples=bp['draws_each'],seed=bp['seed'],alpha=bp['one_sided_alpha_each']);assert interval['valid_replicates']==bp['draws_each']
 if indices is None:indices=idx
 else:assert np.array_equal(indices,idx)
 reps[key]=rep;metrics['comparators'][key]={'full_mean_gain':float(diff.mean()),'annual':annual,'bootstrap':interval};gates[key]={'full_positive':bool(diff.mean()>0),'both_years_positive':bool(all(z['mean_gain']>0 for z in annual)),'both_years_Spearman_positive':bool(all(z['Spearman']>0 for z in annual)),'corrected_lower_positive':bool(interval['lower_bound']>0)}
passed=bool(gates['coefficient_positive'] and all(all(v.values()) for v in gates.values() if isinstance(v,dict)))
with R.start(experiment_name=plan['experiment'],uri=str(card/'qlib_recorder')):
 rec=R.get_recorder();dataset.config(dump_all=True,recursive=True);rec.save_objects(artifact_path='inputs',plan=plan,preflight=preflight,dataset=dataset);rec.save_objects(artifact_path='models',baseline=control,candidate=candidate);rec.save_objects(artifact_path='predictions',scored=scored);rec.save_objects(artifact_path='analysis',full_calendar=full,metrics=metrics,gates=gates,bootstrap_replicates=reps,bootstrap_indices=indices,information_gate_pass=passed);rec.log_metrics(fits=2,native_accounts=0,information_gate_pass=int(passed),gain_vs_baseline=metrics['comparators']['baseline']['full_mean_gain']);rid,eid=rec.id,str(rec.experiment_id)
state['status']='finished';(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
def ready(x):
 if isinstance(x,dict):return {str(k):ready(v) for k,v in x.items()}
 if isinstance(x,(list,tuple)):return [ready(v) for v in x]
 if isinstance(x,np.generic):return x.item()
 return x
summary={'experiment':plan['experiment'],'decision':'conditional_credit_pair_cap_value_information_pass_needs_source_PIT_and_0bp_account' if passed else 'not_promoted_fixed_credit_pair_cap_value_information_failed','objective_achieved':False,'plan_sha256':preflight['plan_sha256'],'preflight':preflight,'metrics':metrics,'gates':gates,'information_gate_pass':passed,'new_model_fits':2,'new_bootstraps':2,'new_native_accounts':0,'historical_first_public_or_revision_qualified':False,'historical_independence':False,'recorder_id':rid,'experiment_id':eid,'recorder_uri':str(card/'qlib_recorder'),'artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['inputs/*','models/*','predictions/*','analysis/*']},'limitations':['This is a local one-step cap-release value screen, not a closed-loop SOXX account.','Yahoo historical dividend and raw-price first-public/revision history is unknown; no PIT certification.','Reused 2021/2022 labels and prior credit H5 research are not independent forward evidence.']};summary=ready(summary);(card/'summary.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2)+'\n')
evidence=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=rid,experiment_id=eid);settings=_settings(root);remote=_ssh(settings['ssh'],'sha256sum '+shlex.quote(evidence['nas_path'])).split()[0];assert remote==evidence['sha256'];summary['NAS_evidence']=evidence;summary['NAS_readback']={'sha256':remote,'verified':True};(card/'summary.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2)+'\n')
print(json.dumps({'decision':summary['decision'],'preflight':preflight,'metrics':metrics,'gates':gates,'recorder_id':rid,'NAS_sha256':remote},ensure_ascii=False,indent=2))


## 中文结论

该固定候选只读取前一交易日收盘可形成的 HYG−LQD 20 交易日总收益，不改变原 Stage1 完整链或 0bp 加 IBKR Tiered 费用口径。以原 Stage1 同状态、向 SOXX 多投最多 0.10 且次日共同复位的局部净价值为接收任务；原 HYG−LQD 五日行情预测固定路线仍关闭。2018–2020 成熟受限训练 239 日，2021/2022 评价 56/245 日，完整评价日历 503 日。历史分红及收盘价首次发布／修订版本尚未认证，本检验仅是条件性开发反馈。

新增标准化系数 **+0.00009214**，符号符合预登记的正向假设。但候选 MSE **6.404552308e-06**，高于匹配四项对照 **6.368773792e-06** 和训练均值 **6.330007046e-06**。完整配对 MSE 改善对两控制分别为 **-3.577851596e-08**、**-7.454526193e-08**；2021 的候选排序 Spearman **-0.1119**，2022 为 **+0.0255**。修正区块下界分别为 **-1.223332984e-07**、**-4.220052509e-07**。除系数方向外，预登记信息门全部失败。

决定 **`not_promoted_fixed_credit_pair_cap_value_information_failed`**。不反转信用变量、不改20日窗、年份、alpha、风险上限或标签定义救援。共2次固定模型拟合、2次区块检验、0新资金账户；原生 Qlib Recorder `bf32b204e6c74cd7a32ccb6ac5b3b814` 已完成读取，NAS SHA-256 `bd81c27bfc1cf867a7168c0ac61df62e85ff95d833140ac2b2b2f8ede001c861` 经远端校验及五项模型／预测／门工件冷读一致。首次 notebook 执行在科学结果与 Recorder 完成后因漏建 `nas_binding.json` 于归档阶段停止；只补绑定并归档/冷读原 Recorder，未重拟合或重跑账户。

局部标签不能相加为完整路径 SOXX 超额。原完整 Stage1 0bp SOXX 硬门、2/5bp 和独立前向证据仍未通过；继续寻找实质不同的决策时信息或机制。
